# LDVAE Implementation

**Project 2: Deep generative models for personalised oncology** · ML for Healthcare, UC3M 2026/27

We performed the following steps:

1. Look at the data and check why we need a model made for counts.
2. Compare one way of representing each cell: **LDVAE**
3. **Question (a):** should we "correct" for the patient? We check this separately for each cell type.
4. **Question (e):** does **LDVAE** generate realistic data? We compare an NB model, a ZINB model and a simple Poisson model.
5. A first look at **question (c)**: can we describe a patient by its mix of cell types?

This is done with a random subset of **30,000 cells**.

## 0. The LDVAE model (Svensson., 2018)

(Acrualizar con LDVAE)


For each cell $n$ and gene $g$:

$$
z_n \sim \mathcal{N}(0, I), \qquad \ell_n \sim \text{LogNormal}(\ell_\mu, \ell_\sigma^2), \qquad \rho_n = f_w(z_n, s_n)
$$
$$
x_{ng} \sim \text{ZINB}\big(\text{mean} = \ell_n \rho_{ng},\; \text{dispersion } \theta_g,\; \text{dropout } f_h^g(z_n, s_n)\big)
$$

In other words:
- $z_n$: a small vector (10 numbers) that summarises the biology of the cell. This is what we use for clustering.
- $\ell_n$: how many molecules were captured in the cell (sequencing depth).
- $s_n$: the batch. If we set the batch to the patient, the model removes differences between patients from $z_n$. The risk is that it also removes real differences between tumours.
- The encoder learns $z_n$ for each cell, and the model is trained by maximising the ELBO.

Things from the paper we want to check on our data:
1. Their metrics: k-means + ARI/NMI, and how well batches mix among the 50 nearest neighbours of each cell.
2. Their result that the simpler **NB** model already explains the zeros, so ZINB is not really needed (section 7).
3. Their warning that batch correction assumes **all batches contain the same biology**. This is not true for tumour cells, because every tumour is different.

In [ ]:
# Install / update dependencies if needed (Colab)
import importlib.util
import pandas as pd
from packaging.version import Version

missing = any(importlib.util.find_spec(m) is None for m in ["anndata", "scanpy", "scvi", "skmisc", "seaborn"])
old_pandas = Version(pd.__version__) < Version("2.3")

if missing or old_pandas:
    !pip install -q scvi-tools scanpy scikit-misc seaborn "pandas>=2.3"
    print("Packages updated. Restarting the session: when it reconnects, run this cell again.")
    import os
    os.kill(os.getpid(), 9)   # restarts the Colab runtime so the new pandas is loaded

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.io
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scanpy as sc
import scvi
import torch

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 1
sc.set_figure_params(dpi=90, frameon=False)

SEED = 0
scvi.settings.seed = SEED
rng = np.random.default_rng(SEED)

print("pandas", pd.__version__, "| scvi-tools", scvi.__version__, "| scanpy", sc.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# ---------------- GLOBAL CONFIG ----------------
DATA_DIR    = Path("data")
H5AD_PATH   = DATA_DIR / "wu2021_brca.h5ad"                # option A: .h5ad from CELLxGENE
GEO_DIR     = DATA_DIR / "Wu_etal_2021_BRCA_scRNASeq"      # option B: extracted GEO GSE176078 folder
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR  = Path("models");  MODELS_DIR.mkdir(exist_ok=True)

N_CELLS      = 30_000   # None = full atlas (~100k). Smaller.
N_HVG        = 2_000
N_LATENT     = 10       # same as the paper; PCA uses the same dimension so the comparison is fair
MAX_EPOCHS   = None     # None = scvi-tools default heuristic
K_NEIGHBORS  = 50       # neighbourhood size for the mixing entropy (as in the paper)
RETRAIN      = False    # True = retrain even if saved models already exist

## 2. Load the data

In [ ]:
import urllib.request
import json
import shutil
import ssl

CXG_COLLECTION_ID = "dea97145-f712-431c-a223-6b5f565f362a"   # "A single-cell and spatially resolved atlas of human breast cancers"
CXG_DATASET_ID    = "9fddb063-056d-4202-8b8a-4b0ee531d3ce"   # full atlas, ~100k cells (~850 MB)

def to_dense(x):
    if sp.issparse(x):
        return x.toarray()
    if hasattr(x, "todense"):          # e.g. arrays from the `sparse` library (GCXS)
        return np.asarray(x.todense())
    return np.asarray(x)

def is_integer_counts(X, n=500):
    sub = to_dense(X[:n])
    return bool(np.all(sub >= 0) and np.allclose(sub, np.round(sub)))

def ssl_context():
    try:
        import certifi
        return ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        return ssl.create_default_context()

def download_from_cellxgene():
    # Ask the CELLxGENE curation API for the current .h5ad link, then stream it to disk
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    api_url = (f"https://api.cellxgene.cziscience.com/curation/v1/collections/"
               f"{CXG_COLLECTION_ID}/datasets/{CXG_DATASET_ID}")
    ctx = ssl_context()
    with urllib.request.urlopen(api_url, context=ctx) as r:
        meta = json.load(r)
    url = next((a["url"] for a in meta.get("assets", []) if a.get("filetype") == "H5AD"), None)
    if url is None:
        raise ValueError("No H5AD file found in the CELLxGENE dataset metadata.")

    print(f"Downloading {url} (~850 MB), this can take a while...")
    tmp_path = H5AD_PATH.with_suffix(".h5ad.part")
    with urllib.request.urlopen(url, context=ctx) as r, open(tmp_path, "wb") as f:
        shutil.copyfileobj(r, f, length=16 * 1024 * 1024)
    tmp_path.rename(H5AD_PATH)          # only rename once the download has finished
    print("Download complete.")

def load_wu2021():
    if not H5AD_PATH.exists() and not GEO_DIR.exists():
        print("Data not found locally, downloading it from CELLxGENE.")
        try:
            download_from_cellxgene()
        except Exception as e:
            raise FileNotFoundError(f"Automatic download failed ({e}). Download the file manually "
                                    f"and save it as {H5AD_PATH} (see section 2).") from e

    if H5AD_PATH.exists():
        adata = sc.read_h5ad(H5AD_PATH)
        if not is_integer_counts(adata.X):
            assert adata.raw is not None, "X is not raw counts and there is no .raw, look for the version with raw counts"
            adata = adata.raw.to_adata()
        if "feature_name" in adata.var:   # CELLxGENE uses Ensembl IDs as the index
            adata.var["ensembl_id"] = adata.var_names
            adata.var_names = adata.var["feature_name"].astype(str)
    elif GEO_DIR.exists():
        X = scipy.io.mmread(GEO_DIR / "count_matrix_sparse.mtx").tocsr()
        genes = pd.read_csv(GEO_DIR / "count_matrix_genes.tsv", header=None, sep="\t")[0].astype(str).values
        barcodes = pd.read_csv(GEO_DIR / "count_matrix_barcodes.tsv", header=None, sep="\t")[0].astype(str).values
        if X.shape[0] == len(genes) and X.shape[1] == len(barcodes):
            X = X.T.tocsr()            # GEO stores it as genes × cells
        meta = pd.read_csv(GEO_DIR / "metadata.csv", index_col=0)
        adata = ad.AnnData(X=X, obs=meta.loc[barcodes].copy(), var=pd.DataFrame(index=genes))
    else:
        raise FileNotFoundError("Data not found. See the instructions in section 2.")

    adata.var_names_make_unique()
    assert is_integer_counts(adata.X), "The matrix doesn't look like raw counts"
    return adata

def first_col(obs, candidates):
    for c in candidates:
        if c in obs.columns:
            return c
    raise KeyError(f"None of {candidates} found in obs: {list(obs.columns)}")

adata = load_wu2021()
COLMAP = {
    "patient":        first_col(adata.obs, ["orig.ident", "donor_id", "Patient", "patient"]),
    "subtype":        first_col(adata.obs, ["subtype", "Subtype", "clinical_subtype", "disease"]),
    "celltype":       first_col(adata.obs, ["celltype_major", "author_cell_type", "cell_type"]),
    "celltype_minor": first_col(adata.obs, ["celltype_minor", "celltype_major", "author_cell_type", "cell_type"]),
}
for std, orig in COLMAP.items():
    adata.obs[std] = adata.obs[orig].astype(str).astype("category")
print(COLMAP)

{'patient': 'donor_id', 'subtype': 'subtype', 'celltype': 'celltype_major', 'celltype_minor': 'celltype_minor'}


SUBSAMPLING

In [1]:
if N_CELLS is not None and adata.n_obs > N_CELLS:
    sc.pp.subsample(adata, n_obs=N_CELLS, random_state=SEED)
print(f"{adata.n_obs:,} cells × {adata.n_vars:,} genes | {adata.obs.patient.nunique()} patients")

NameError: name 'N_CELLS' is not defined

WE HAVE ALREADY DONE EXPLORATORY ANALYSIS

The conslussion are:



## 3. Preprocess:

Taking into account the conslussions we have achieved before...

- We keep the raw counts in `layers["counts"]` (this is what scVI uses).
- `X` is normalised and log-transformed (this is what PCA uses).
- We keep the **2,000 most variable genes** (HVGs).

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=10)
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

try:
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3",
                                layer="counts", batch_key="patient", subset=True)
except Exception as e:  # patients with very few cells can make the loess fit fail
    print("Per-patient HVG selection failed, using global HVGs instead:", e)
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3", layer="counts", subset=True)
adata

Per-patient HVG selection failed, using global HVGs instead: b'reciprocal condition number  1.2231e-14'


AnnData object with n_obs × n_vars = 30000 × 2000
    obs: 'donor_id', 'percent_mito', 'nCount_RNA', 'nFeature_RNA', 'celltype_major', 'celltype_minor', 'celltype_subset', 'subtype', 'gene_module', 'calls', 'normal_cell_call', 'CNA_value', 'batch_run', 'multiplexed', 'cryo_state', 'development_stage_ontology_term_id', 'cancer_type', 'ER', 'PR', 'HER2_IHC', 'HER2_ISH', 'HER2_ISH_ratio', 'Ki67', 'subtype_by_IHC', 'treatment_status', 'treatment_details', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'disease_ontology_term_id', 'grade', 'cell_type_ontology_term_id', 'tissue_type', 'HER2_ISH_legacy', 'cell_type_ontology_term_id_legacy', 'celltype_major_legacy', 'celltype_minor_legacy', 'celltype_subset_legacy', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'patient', 'celltype', 'n_genes'
    var: 'cellrang

In [ ]:
# Es necesario?

# Registrar los datos de expresión génica
scvi.model.LinearSCVI.setup_anndata(adata,layer="counts")

In [ ]:
def train_ldvae(adata, name, batch_key, gene_likelihood, n_latent=N_LATENT, **model_kwargs):
    a = adata.copy()                      # each model gets its own AnnData registration
    scvi.model.SCVI.setup_anndata(a, layer="counts", batch_key=batch_key)
    path = 
    if path.exists() and not RETRAIN:
        model = scvi.model.SCVI.load(path, adata=a)
    else:
        model = scvi.model.SCVI(a, n_latent=n_latent, gene_likelihood=gene_likelihood, **model_kwargs)
        model = scvi.model.LinearSCVI(a,n_latent=n_latent, gene_likelihood=gene_likelihood, **model_kwargs)
        latent = model.get_latent_representation()
        model.train(max_epochs=MAX_EPOCHS, early_stopping=True, check_val_every_n_epoch=1)
        model.save(path, overwrite=True)
    return model, a


In [ ]:
# Training curves: validation ELBO (has it converged? is it overfitting?)
fig, ax = plt.subplots(figsize=(7, 4))
for name, (model, _) in MODELS.items():
    h = model.history
    if "elbo_validation" in h:
        ax.plot(h["elbo_validation"].values.ravel(), label=name)
ax.set(xlabel="epoch", ylabel="ELBO (validation)", title="Convergence"); ax.legend(); plt.show()